# DSCI552: Final Project
- Name: Andrea Fernandez Cruz
- Github Username: ndreaelizabth
- USC ID: 6735339003


In [13]:
from pathlib import Path
import os
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Setup imports complete.")
print("Current folder:", Path.cwd())

Setup imports complete.
Current folder: /workspaces/dsci552_finalproject_spring_2026/notebook


In [14]:
PROJECT_ROOT = Path("..")

# Your dataset may be either data/C1...C5 or data/defungi/C1...C5
if (PROJECT_ROOT / "data" / "defungi").exists():
    DATA_DIR = PROJECT_ROOT / "data" / "defungi"
else:
    DATA_DIR = PROJECT_ROOT / "data"

SAVED_MODELS_DIR = Path("saved_models")
PLOTS_DIR = Path("plots")
RESULTS_DIR = Path("results")

SAVED_MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

classes = sorted([folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()])

print("Classes found:", classes)
print("Number of classes:", len(classes))

FileNotFoundError: [Errno 2] No such file or directory: '../data'

In [ ]:
image_counts = {}

for class_name in classes:
    class_folder = DATA_DIR / class_name
    
    image_files = [
        file for file in class_folder.iterdir()
        if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]
    
    image_counts[class_name] = len(image_files)

image_counts_df = pd.DataFrame.from_dict(
    image_counts,
    orient="index",
    columns=["Number of Images"]
)

image_counts_df

,Number of Images
C1,4404
C2,2334
C3,819
C4,818
C5,739


In [ ]:
SPLIT_DIR = PROJECT_ROOT / "data" / "split"

TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"
TEST_DIR = SPLIT_DIR / "test"

print("Split folder will be:", SPLIT_DIR)

NameError: name 'PROJECT_ROOT' is not defined

In [ ]:
def split_dataset(source_dir, train_dir, val_dir, test_dir, seed=42):
    random.seed(seed)

    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    for class_name in classes:
        class_path = source_dir / class_name

        image_paths = [
            file for file in class_path.iterdir()
            if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ]

        random.shuffle(image_paths)

        total = len(image_paths)

        train_end = int(total * 0.75)
        val_end = train_end + int(total * 0.15)

        train_images = image_paths[:train_end]
        val_images = image_paths[train_end:val_end]
        test_images = image_paths[val_end:]

        split_info = [
            (train_images, train_dir),
            (val_images, val_dir),
            (test_images, test_dir)
        ]

        for images, split_folder in split_info:
            class_split_folder = split_folder / class_name
            class_split_folder.mkdir(parents=True, exist_ok=True)

            for image_path in images:
                destination = class_split_folder / image_path.name

                if not destination.exists():
                    shutil.copy(image_path, destination)

        print(
            f"{class_name}: "
            f"train={len(train_images)}, "
            f"validation={len(val_images)}, "
            f"test={len(test_images)}, "
            f"total={total}"
        )

In [ ]:
split_dataset(DATA_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR)

C1: train=3303, validation=660, test=441, total=4404
C2: train=1750, validation=350, test=234, total=2334
C3: train=614, validation=122, test=83, total=819
C4: train=613, validation=122, test=83, total=818
C5: train=554, validation=110, test=75, total=739


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

print("TensorFlow version:", tf.__version__)
print("Keras is working.")

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)

I0000 00:00:1778207106.727686    4894 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778207107.140573    4894 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778207109.145624    4894 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0
Keras is working.
GPUs found: []


E0000 00:00:1778207109.569198    4894 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names

print("Class names:", class_names)
print("Number of classes:", len(class_names))

NameError: name 'TRAIN_DIR' is not defined